In [ ]:
# 사전설치 : pip install kss torch langchain-google-genai langchain gradio transformers
# 불필요한 캐시 삭제 : pip cache purge
# 허깅페이스 다운 가속화 : pip install hf_xet
# 허깅페이스 로그인 접속 : pip install huggingface_hub, 터미널에서 huggingface-cli login, settings> Access Token에서 토큰 생성 후 액세스 토큰 입력
import os
import re  # 문자열을 규칙으로 검색·판별·치환하기 위한 도구
import numpy as np
import warnings
import kss  # 한국어 문장 분리
import gradio as gr
from transformers import pipeline
from langchain_google_genai import ChatGoogleGenerativeAI
import warnings
warnings.filterwarnings('ignore')   # 파이썬의 warnings 모듈을 사용하여 발생하는 경고 메시지를 무시

In [ ]:
# Gemini API 키 설정
os.environ["GOOGLE_API_KEY"] = "insert your api key"  # 발급받은 Gemini API 키

In [ ]:
# NewsAnalyzer
# ├─ __init__                      → 모델 초기화 (Gemini, FinBert, Zero-Shot) 및 레이블 정의
# ├─ analyze_chunk_fake_news       → [핵심] 텍스트 조각(Chunk) 단위 가짜 뉴스 확률 계산 (Zero-Shot)
# ├─ check_fake_news_indicators    → [규칙] 가짜 뉴스 패턴(과장, 기적, 형식 등) 검사
# ├─ adjust_score                  → [보정] AI 점수에 규칙 기반 가중치 합산
# ├─ analyze_with_gemini           → [보조] Gemini를 이용한 팩트체크 및 3줄 요약
# ├─ analyze_sentiment             → [보조] 기사의 전반적인 감성/분위기 분석
# └─ analyze_news                  → [메인] 문장 분리(KSS), 전체 분석 루프 실행 및 결과 리포트 생성

class NewsAnalyzer:
    def __init__(self):
        print("시스템 초기화 및 모델 로딩 중...")

        # 1. Gemini 모델 (보조 분석용)
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0
        )

        # 2. 감성 분석 모델
        self.sentiment_analyzer = pipeline(
            "sentiment-analysis",
            model="snunlp/KR-FinBert-SC",
            tokenizer="snunlp/KR-FinBert-SC"
        )

        # 3. 가짜 뉴스 분류 모델 (Zero-Shot)
        self.fake_news_classifier = pipeline(
            "zero-shot-classification",
            model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
            device=-1  # 모델 CPU/GPU 실행 구분 : CPU(-1), 1번째 GPU(0), 두번째 GPU(1)
        )

        # 레이블 세분화: 단순 '거짓'이 아니라 구체적인 유형을 제시하여 포착률 높임
        self.target_labels = [
            "객관적인 사실 전달",  # 긍정 레이블
            "허구의 지어낸 이야기", # 부정 레이블 1
            "과장된 광고 및 홍보",  # 부정 레이블 2
            "검증되지 않은 루머",  # 부정 레이블 3
            "비과학적인 음모론"    # 부정 레이블 4
        ]

    def analyze_chunk_fake_news(self, text):
        """다각도 레이블링을 통한 가짜 뉴스 점수 산출"""
        try:
            # Zero-shot 분류 수행
            result = self.fake_news_classifier(
                text[:512],
                self.target_labels,
                multi_label=False # 가장 적합한 하나를 고르게 함 (확률 총합 1)
            )

            labels = result['labels']
            scores = result['scores']

            # '객관적인 사실 전달'을 제외한 나머지 부정적인 레이블들의 점수 합계 계산
            # 즉, (허구 + 광고 + 루머 + 음모론)의 확률이 가짜 뉴스 점수가 됨
            fake_prob = 0.0
            for label, score in zip(labels, scores):
                if label != "객관적인 사실 전달":
                    fake_prob += score

            return fake_prob
        except Exception as e:
            print(f"분류 오류: {e}")
            return 0.5  # 애매할 경우 일단 중간점수로 return

    def check_fake_news_indicators(self, text):
        """규칙 기반 탐지 강화"""
        # 정규표현식 등을 활용한 키워드 검사
        indicators = {
            # 1. 과도한 문장부호 (?! 반복)
            'excessive_punctuation': len(re.findall(r'[!?]{2,}', text)) > 0,

            # 2. 낚시성/선정적 단어 (cf. any함수: 하나라도 true이면 true, 전부 false면 false)
            'sensational_words': any(w in text for w in ['충격', '경악', '속보', '특종', '단독', '극비', '놀랍게도']),

            # 3. 사이비/기적/과장 광고 키워드
            'miracle_claims': any(w in text for w in ['기적', '완치', '불로장생', '영원한', '되찾은', '마법', '비법', '천기누설']),

            # 4. 출처 불분명
            'unverified_sources': any(p in text for p in ['카더라', '소식통', '알려졌다', '전해진다', '들려오고']),

            # 5. 뉴스에 어울리지 않는 포맷 (마크다운 볼드체 등)
            'weird_formatting': len(re.findall(r'\*\*.*?\*\*', text)) > 0    # \*\* : 실제문자 **   .*? : 아무 문자 0개 이상
        }
        return indicators

    def adjust_score(self, base_score, indicators):
        """발견된 특징에 따라 점수 가중치 부여"""
        score = base_score

        # 감점 요인별 가중치 (기본 모델 점수에 추가)
        if indicators['excessive_punctuation']: score += 0.1
        if indicators['sensational_words']: score += 0.15
        if indicators['miracle_claims']: score += 0.20
        if indicators['unverified_sources']: score += 0.10
        if indicators['weird_formatting']: score += 0.10

        return min(max(score, 0), 1.0) # 0~1 사이로 제한

    def analyze_with_gemini(self, article):
        """Gemini 텍스트 분석"""
        try:
            prompt = (
                f"다음 텍스트가 가짜 뉴스인지 팩트체크해줘. 특히 과학적으로 불가능한 내용이 있는지 확인해줘.\n"
                f"3줄 요약과 함께 진위 여부를 냉정하게 평가해.\n\n내용:\n{article[:2000]}"
            )
            response = self.llm.invoke(prompt)
            return response.content
        except Exception:
            return "Gemini 분석 불가 (API Key 확인)"

    def analyze_sentiment(self, article):
        """기사의 전반적인 감성/분위기 분석"""
        try:
            # 기사의 앞부분 512자만 사용하여 빠르게 분석
            result = self.sentiment_analyzer(article[:512])[0]
            label = result['label']
            score = result['score']
            return f"{label} (강도: {score:.2f})"
        except Exception as e:
            # print(f"감성 분석 오류: {e}")
            return "분석 불가"

    def analyze_news(self, article):
        """통합 분석"""
        if not article.strip(): return "내용을 입력해주세요."

        # 1. 텍스트 청킹
        sentences = kss.split_sentences(article)
        chunks = []
        current_chunk = ""
        for sent in sentences:
            if len(current_chunk) + len(sent) < 500:
                current_chunk += sent + " "
            else:
                chunks.append(current_chunk)
                current_chunk = sent + " "
        if current_chunk: chunks.append(current_chunk)

        # 2. 청크별 분석 수행
        chunk_scores = []
        for chunk in chunks:
            ai_score = self.analyze_chunk_fake_news(chunk) # 모델 점수
            indicators = self.check_fake_news_indicators(chunk) # 규칙 검사
            final_score = self.adjust_score(ai_score, indicators) # 점수 보정
            chunk_scores.append(final_score)

        # 평균 점수 산출
        avg_fake_prob = np.mean(chunk_scores) if chunk_scores else 0.5

        # [판정 기준] 50% 이상이면 의심, 70% 이상이면 확실한 가짜
        if avg_fake_prob >= 0.7:
            verdict = "확실한 가짜 뉴스/허위 정보"
        elif avg_fake_prob >= 0.5:
            verdict = "가짜 뉴스 의심 (주의 필요)"
        else:
            verdict = "신뢰할 수 있는 뉴스 가능성 높음"

        # 3. 보조 분석
        gemini_result = self.analyze_with_gemini(article)

        # 감성 분석
        sentiment_text = self.analyze_sentiment(article)

        return f"""
=== 뉴스 진위 분석 리포트 ===

[최종 판정]
{verdict}
(가짜/허구 확률: {avg_fake_prob * 100:.2f}%)

[AI 상세 분석]
1. 내용 신뢰도 평가
   - 문맥상 '허구/과장/루머'일 확률이 {avg_fake_prob * 100:.2f}% 로 계산되었습니다.
   - 텍스트 분위기: {sentiment_text}

2. Gemini 팩트체크 의견
{gemini_result}
"""

In [ ]:
def create_interface():
    analyzer = NewsAnalyzer()

    iface = gr.Interface(
        fn=analyzer.analyze_news,
        inputs=gr.Textbox(lines=10, placeholder="뉴스 기사 내용을 입력하세요..."),
        outputs=gr.Textbox(lines=20),
        title="가짜 뉴스 판별 시스템",
        description="AI 모델을 활용한 뉴스 진위 판별 시스템"
    )
    return iface

In [ ]:
if __name__ == "__main__":
    analyzer = NewsAnalyzer()
    iface = create_interface()
    iface.launch()

모델 초기화 중...
감성 분석 모델 로딩...


Device set to use cpu


가짜 뉴스 분류 모델 로딩...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


모델 초기화 완료!
모델 초기화 중...
감성 분석 모델 로딩...


Device set to use cpu


가짜 뉴스 분류 모델 로딩...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


모델 초기화 완료!
* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [ ]:
iface.close()

Closing server running on port: 7860
